# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 2: *Data Merging*
##### Version Number: 3.0
---
### Contents  
> 1. *Merge sample points with weather data*
> 2. *Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*
> 3. *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from extracted from gridMET.
### Inputs
- Daily Weather Readings - `gridmet_final.csv` 
- Wildfire Damage Data - `fire_data.csv` (cleaned in module 1)
- California Mesh Sampling Grid - `Sampling_grids.csv` (built in appendix A) 
---
### Outputs  
- `model_fire_pop_income.csv` Cleaned weather dataset merged with fire damage severity, population and mean income data
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

---

### Data Loading and Exploration

In [3]:
weather = pd.read_csv('../data/raw/gridmet_final.csv')
fire_data = pd.read_csv("../data/processed/fire_data.csv")
samples = pd.read_csv("../data/processed/cleaned_grids.csv")

In [4]:
samples

,grid_id,centroid_easting,centroid_northing,influence_zone,interface_zone,intermix_zone,dominant_province_description,dominant_province_percent,sum_province_area,dominant_section_description,...,power_line_meters,power_line_density,total_housing,total_population,housing_density,population_density,maximum_x,minimum_y,maximum_y,minimum_x
0,2,240898.9343,-582327.0481,3.462695e+07,4.014329e+07,1.194620e+07,California Coastal Chaparral Forest and Shrub,10.382990,2.197041e+08,Southern California Coast,...,134443.531055,0.027654,132287.539814,2.964600e+05,0.027211,0.060981,-117.180758,32.528595,32.955056,-117.683965
1,3,286898.9343,-582327.0481,7.500688e+08,2.464481e+08,1.577920e+08,California Coastal Chaparral Forest and Shrub,64.913231,1.901271e+09,Southern California Coast,...,895418.608453,0.045281,655216.214990,1.801619e+06,0.033134,0.091106,-116.689706,32.515235,32.943786,-117.195291
2,4,332898.9343,-582327.0481,6.136700e+08,7.307676e+06,6.939645e+07,California Coastal Range Open Woodland-Shrub-C...,78.522516,1.691828e+09,Southern California Mountains and Valleys,...,231134.844287,0.013630,5472.499438,1.313075e+04,0.000323,0.000774,-116.198831,32.499735,32.930364,-116.706767
3,5,378898.9343,-582327.0481,5.681297e+07,1.034100e+06,8.595000e+05,American Semi-Desert and Desert,47.181032,1.412266e+09,Colorado Desert,...,156790.788050,0.011104,713.471288,4.510455e+03,0.000051,0.000319,-115.708160,32.482097,32.914794,-116.218417
4,6,424898.9343,-582327.0481,8.379900e+06,5.679000e+05,4.932001e+05,American Semi-Desert and Desert,53.284989,1.127510e+09,Colorado Desert,...,562387.632567,0.049884,39478.063283,1.273995e+05,0.003502,0.011300,-115.217720,32.462321,32.897075,-115.730268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,243,-173101.0657,429672.9519,9.795318e+07,5.634000e+05,2.543400e+06,Sierran Steppe-Mixed Forest-Coniferous Forest-...,100.000000,2.116000e+09,Southern Cascades,...,97380.573665,0.005527,941.230533,1.811330e+03,0.000053,0.000103,-121.799799,41.655609,42.079489,-122.364917
232,244,-127101.0657,429672.9519,1.172576e+07,1.449000e+05,8.217000e+05,Sierran Steppe-Mixed Forest-Coniferous Forest-...,100.000000,2.116000e+09,Modoc Plateau,...,263139.055725,0.015605,920.940001,2.021440e+03,0.000055,0.000120,-121.248314,41.664688,42.086199,-121.810330
233,245,-81101.0657,429672.9519,0.000000e+00,0.000000e+00,0.000000e+00,Sierran Steppe-Mixed Forest-Coniferous Forest-...,100.000000,2.116000e+09,Modoc Plateau,...,887.694414,0.000054,0.000000,0.000000e+00,0.000000,0.000000,-120.696745,41.671355,42.090481,-121.255619
234,246,-35101.0657,429672.9519,4.417731e+07,5.877000e+05,2.214804e+06,Sierran Steppe-Mixed Forest-Coniferous Forest-...,94.785264,2.116000e+09,Modoc Plateau,...,74521.828841,0.004614,332.412470,5.144503e+02,0.000021,0.000032,-120.145128,41.675610,42.092334,-120.700822


## 1. Merge sample points with weather data

In [5]:
samples['geometry'] = [Point(xy) for xy in zip(samples['centroid_easting'], samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(samples, geometry='geometry', crs="EPSG:3310")

#samples_gdf = samples_gdf.to_crs('4326')

## generate unique identifier
#samples_gdf['Sample_ID'] = range(1, len(samples_gdf) + 1)

## fix error from ArcGIS
#samples_gdf['County'] = samples_gdf['County'].replace('South Coast Valleys', 'San Diego')

In [6]:
sample_ids = samples['Sample_ID'].unique()

In [7]:
sample_ids

array([  1,   2,   3,   4,   5,   6,  13,   7,   8,   9,  10,  11,  12,
        14,  15,  16,  17,  21,  22,  18,  19,  20,  32,  33,  23,  24,
        25,  26,  27,  28,  29,  30,  31,  44,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        89,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 109, 110, 111, 112, 113, 114, 115, 116, 117, 119,
       120, 121, 122, 123, 124, 125, 126, 128, 129, 130, 131, 132, 133,
       134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 146, 147, 173,
       148, 149, 150, 151, 152, 154, 155, 159, 160, 162, 163, 164, 156,
       157, 158, 165, 166, 167, 168, 169, 171, 172], dtype=int64)

In [8]:
weather

,Sample_ID,Date,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,...,Daily Minimum Air Temperature,Daily Maximum Air Temperature,Vapor Pressure Deficit,Wind Speed,Standardized Precipitation Index 30-Day,Standardized Precipitation Index 180-Day,Standardized Precipitation Evapotranspiration Index 30-Day,Standardized Precipitation Evapotranspiration Index 90-Day,Standardized Precipitation Evapotranspiration Index 180-Day,Palmer Drought Severity Index
0,1,2018-01-01,24.0,37.0,1.7,16.600000,13.700000,0.0,97.800003,52.700001,...,283.000000,290.700012,0.41,1.3,-2.09,-2.09,-2.09,-2.09,-2.09,-2.390000
1,1,2018-01-02,25.0,39.0,2.0,15.900000,13.800000,0.0,90.599998,43.200001,...,285.299988,294.399994,0.68,1.2,-2.09,-2.09,-2.09,-2.09,-2.09,-2.390000
2,1,2018-01-03,23.0,36.0,1.6,15.900000,13.800000,0.0,93.500000,53.400002,...,286.000000,292.899994,0.50,1.2,-1.69,-2.09,-2.09,-2.09,-2.09,-2.550000
3,1,2018-01-04,27.0,38.0,2.3,16.299999,14.100000,0.0,98.300003,47.700001,...,285.000000,294.200012,0.56,1.9,-1.69,-2.09,-2.09,-2.09,-2.09,-2.550000
4,1,2018-01-05,31.0,38.0,3.1,16.000000,14.200000,0.0,89.699997,46.299999,...,286.500000,294.600006,0.67,3.0,-1.69,-2.09,-2.09,-2.09,-2.09,-2.550000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446335,173,2025-01-19,23.0,28.0,1.7,12.600000,20.700001,0.0,76.199997,28.600000,...,262.899994,276.799988,0.31,2.7,1.01,0.90,0.90,0.90,0.63,1.219999
446336,173,2025-01-20,26.0,29.0,1.8,12.600000,20.200001,0.0,77.000000,26.700001,...,260.500000,275.200012,0.28,3.3,1.01,0.90,0.90,0.90,0.63,1.219999
446337,173,2025-01-21,27.0,32.0,2.4,11.300000,19.700001,0.0,53.599998,20.200001,...,265.500000,279.000000,0.45,3.0,1.01,0.90,0.90,0.90,0.63,1.219999
446338,173,2025-01-22,26.0,34.0,2.3,10.300000,19.299999,0.0,51.900002,18.799999,...,266.299988,280.399994,0.50,2.5,1.01,0.90,0.90,0.90,0.63,1.219999


In [9]:
newweather = weather[weather['Sample_ID'].isin(sample_ids)]

In [10]:
newweather['Sample_ID'].unique()

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 109, 110, 111, 112, 113, 114, 115, 116, 117, 119,
       120, 121, 122, 123, 124, 125, 126, 128, 129, 130, 131, 132, 133,
       134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 146, 147, 148,
       149, 150, 151, 152, 154, 155, 156, 157, 158, 159, 160, 162, 163,
       164, 165, 166, 167, 168, 169, 171, 172, 173], dtype=int64)

In [11]:
newweather

,Sample_ID,Date,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,...,Daily Minimum Air Temperature,Daily Maximum Air Temperature,Vapor Pressure Deficit,Wind Speed,Standardized Precipitation Index 30-Day,Standardized Precipitation Index 180-Day,Standardized Precipitation Evapotranspiration Index 30-Day,Standardized Precipitation Evapotranspiration Index 90-Day,Standardized Precipitation Evapotranspiration Index 180-Day,Palmer Drought Severity Index
0,1,2018-01-01,24.0,37.0,1.7,16.600000,13.700000,0.0,97.800003,52.700001,...,283.000000,290.700012,0.41,1.3,-2.09,-2.09,-2.09,-2.09,-2.09,-2.390000
1,1,2018-01-02,25.0,39.0,2.0,15.900000,13.800000,0.0,90.599998,43.200001,...,285.299988,294.399994,0.68,1.2,-2.09,-2.09,-2.09,-2.09,-2.09,-2.390000
2,1,2018-01-03,23.0,36.0,1.6,15.900000,13.800000,0.0,93.500000,53.400002,...,286.000000,292.899994,0.50,1.2,-1.69,-2.09,-2.09,-2.09,-2.09,-2.550000
3,1,2018-01-04,27.0,38.0,2.3,16.299999,14.100000,0.0,98.300003,47.700001,...,285.000000,294.200012,0.56,1.9,-1.69,-2.09,-2.09,-2.09,-2.09,-2.550000
4,1,2018-01-05,31.0,38.0,3.1,16.000000,14.200000,0.0,89.699997,46.299999,...,286.500000,294.600006,0.67,3.0,-1.69,-2.09,-2.09,-2.09,-2.09,-2.550000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446335,173,2025-01-19,23.0,28.0,1.7,12.600000,20.700001,0.0,76.199997,28.600000,...,262.899994,276.799988,0.31,2.7,1.01,0.90,0.90,0.90,0.63,1.219999
446336,173,2025-01-20,26.0,29.0,1.8,12.600000,20.200001,0.0,77.000000,26.700001,...,260.500000,275.200012,0.28,3.3,1.01,0.90,0.90,0.90,0.63,1.219999
446337,173,2025-01-21,27.0,32.0,2.4,11.300000,19.700001,0.0,53.599998,20.200001,...,265.500000,279.000000,0.45,3.0,1.01,0.90,0.90,0.90,0.63,1.219999
446338,173,2025-01-22,26.0,34.0,2.3,10.300000,19.299999,0.0,51.900002,18.799999,...,266.299988,280.399994,0.50,2.5,1.01,0.90,0.90,0.90,0.63,1.219999


In [12]:
# Merge on BOTH station and date
samples_daily = newweather.merge(
    samples_gdf,
    on=['Sample_ID'],
    how='left'
)

### Reorganize so Sample ID and Date are first two columns
#samples_daily.insert(0, 'Date', samples_daily.pop('Date'))
#samples_daily.insert(0, 'Sample_ID', samples_daily.pop('Sample_ID'))

In [13]:
post_merge_check (samples_daily, weather)

Premerged shape:  (446340, 22)
Merged shape:  (608880, 63)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


## 2. Spatial Join of Nearest Sampling Locations with Fire Damage Datasets

To prepare for feature engineering, we spatially join each fire location to its nearest sampling point. This enables us to associate daily environmental conditions with each fire based on geographic proximity, rather than relying solely on county or administrative boundaries.

In [14]:
fire_data['geometry'] = [Point(xy) for xy in zip(fire_data['Fire_Longitude'], fire_data['Fire_Latitude'])]
fire_data_gdf = gpd.GeoDataFrame(fire_data, geometry='geometry', crs="EPSG:4326")

samples_daily_gdf = gpd.GeoDataFrame(samples_daily, geometry='geometry', crs="EPSG:3310")

## Project to Equal area projection for more accuracy
samples_projected = samples_daily_gdf.to_crs(3310)
fire_data_projected = fire_data_gdf.to_crs(3310)

In [15]:
## Buffer distance in meters, Sampling grids are located approximately 47163 meters apart
buffer_dist = 47163

In [16]:
## Initialize variables to track total damage and number of fires associated with each point
samples_projected['fire_count'] = 0
fire_data_projected['total_fire_damage'] = 0.0

#### Main loop of Spatial Join algorithm
- Loop through all dates in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move to next date
- Load sampling points associated with current date
- Create a buffer, sized as defined earlier, around each fire for current date
- Intersection spatial join of sampling points and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - Total estimated cost is summed for all fires in range
- Assign samples to dataframe

In [17]:
for dt in fire_data_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_data_projected[fire_data_projected['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_projected[samples_projected['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each fire
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered.buffer(buffer_dist)

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    # Aggregate counts and total damage per sample
    grouped = joined.groupby('grid_id').agg(
        fire_count=('Estimated Damage', 'count'),
        total_fire_damage=('Estimated Damage', 'sum'),
        acres = ('Acres', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_projected.loc[samples_projected['Date'] == dt, 'fire_count'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'grid_id'].map(grouped['fire_count']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'total_fire_damage'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'grid_id'].map(grouped['total_fire_damage']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'acres'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'grid_id'].map(grouped['acres']).fillna(0)

In [18]:
post_merge_check(samples_projected, samples_daily)

Premerged shape:  (608880, 63)
Merged shape:  (608880, 66)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  12272


In [19]:
samples_projected.isna().sum()

Sample_ID                       0
Date                            0
Burning Index                   0
Energy Release Component        0
Actual Evapotranspiration       0
                             ... 
minimum_x                       0
geometry                        0
fire_count                      0
total_fire_damage            6136
acres                        6136
Length: 66, dtype: int64

NA values are for buffered areas without any fires in range within the date range. Fill these values with 0. 

In [20]:
samples_projected['total_fire_damage'] = samples_projected['total_fire_damage'].fillna(0)
samples_projected['acres'] = samples_projected['acres'].fillna(0)

#### Clean Dataframe

In [21]:
## sort dataframe by date
samples_projected = samples_projected.sort_values(by='Date')

## reset index
samples_projected = samples_projected.reset_index(drop=True)

---

## 3. Export File

In [22]:
samples_projected.to_csv('../data/processed/samples_projected.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
